# Ablation Row 1 — Baseline (HOG-coarse only, single scale)

| | Value |
|---|---|
| Features | HOG-coarse only on 3 HSV channels (~5292-d raw) |
| PCA | 1215 components (90% variance, Gram-trick) |
| Scales | (1.0,) — single scale |
| HNM | off |
| Threshold | `cfg.detection_threshold = 1.5` (default) |

Artifacts → `microglia-artifacts-row1/`. Dalal & Triggs–style minimal pipeline.

## 1  Imports + Config

In [1]:
import os
import joblib
import numpy as np
import matplotlib.image as mpimg

from pipeline import (
    Config,
    extract_raw_features, fit_pipeline,
    tune_svm, train_svm, evaluate_classifier,
    process_image, multi_scale_detect, non_max_suppression,
    load_gt_boxes, evaluate_detections, plot_gt_vs_pred,
    save_hard_negatives, tune_detection_threshold,
    ensure_test_patches, patch_level_test_eval, detection_level_test_eval,
    extract_features, list_images,
)

%matplotlib inline

In [ ]:
cfg = Config(
    artifact_dir='./microglia-artifacts-row1',
    feature_mode='hog_coarse',
    scale_factors=(1.0,),
    pca_n_components=1215,  # 90% variance — Gram-trick curve, 2026-05-23
    detection_threshold=1.5,
)
os.makedirs(cfg.artifact_dir, exist_ok=True)
print(cfg)

## 2  Train (features → PCA → tune C → fit SVM)

In [ ]:
# Train/Validate patches + HOG-coarse features are already on disk from the
# initial extraction run. To rebuild from source uncomment:
# from pipeline import preprocess_all_images
# train_paths, val_paths = preprocess_all_images(cfg)
# X_train_raw, X_val_raw, y_train_raw, y_val_raw = extract_raw_features(cfg)
# np.savez_compressed(cfg.features_cache,
#                     X_train_raw=X_train_raw, X_val_raw=X_val_raw,
#                     y_train_raw=y_train_raw, y_val_raw=y_val_raw)

# ── Load cached features ───────────────────────────────────────────
cache       = np.load(cfg.features_cache)
X_train_raw = cache["X_train_raw"]
X_val_raw   = cache["X_val_raw"]
y_train_raw = cache["y_train_raw"]
y_val_raw   = cache["y_val_raw"]
print(f"Loaded cache: X_train_raw={X_train_raw.shape}, X_val_raw={X_val_raw.shape}")

# ── Fit scaler + PCA(1215) → tune C → train SVM ─────────────────────────
X_train, X_val, y_train, y_val, scaler, pca = fit_pipeline(
    X_train_raw, X_val_raw, y_train_raw, y_val_raw, cfg
)

best_params = tune_svm(X_train, y_train, cfg)
svm_clf     = train_svm(X_train, y_train, cfg)

joblib.dump(svm_clf, cfg.svm_clf_path)
joblib.dump(scaler,  cfg.scaler_path)
joblib.dump(pca,     cfg.pca_path)
print(f"Saved artifacts → {cfg.artifact_dir}/")

print("\nValidate patch-level metrics:")
evaluate_classifier(svm_clf, X_val, y_val, label='val')


## 3  Held-out test evaluation

In [ ]:
ensure_test_patches(cfg)

print("── Patch-level test ──")
patch_level_test_eval(svm_clf, scaler, pca, cfg)

print("\n── Detection-level test (single scale, t=%.2f) ──" % cfg.detection_threshold)
row1_metrics = detection_level_test_eval(svm_clf, scaler, pca, cfg, show_plots=True)